# NB 0 — Setup & Data Load

> **Yol Haritası adımları:** 0.1 → 0.2 → 0.3
>
> **Amaç:** Çalışma ortamını kur, ham Excel'i bir kez oku, hızlı erişim için
> Parquet'e çevir ve HCP veri spesifikasyonunu (kod sözlüğünü) hazırla.
>
> **Bu notebook'un çıktıları:**
> 1. `data/processed/hcp.parquet` — sonraki tüm notebook'lar buradan okuyacak
> 2. `reports/code_dictionary.json` — HCP veri sözlüğü (kod → anlam eşlemesi)
> 3. `reports/episode_spec.csv` — Episode sayfasının düz tablo hali
> 4. Bir sonraki notebook (NB1) için **birim teyidi** (charge sent mi dolar mı?)

---

### Neden bu notebook ayrı?
Yükleme + sözlük çıkarma işi pahalı (Excel okuma ~30 sn). Bunu **1 kez** yapıp
sonuçları kaydediyoruz; sonraki notebook'lar (`01_..` `02_..` `03_..` `04_..`)
bu dosyalardan okuyacak ve saniyeler içinde açılacak. Bu, profesyonel veri
bilimi projesinin temel disiplinidir: **ham veriye dokunan tek bir yer olsun.**


## 0.1 — Ortam Kurulumu & Kütüphane Sürümleri

**Yapılanlar:** Kütüphaneleri import ediyoruz ve sürümlerini yazıyoruz.

**Neden sürümleri yazdırıyoruz?** Çünkü 6 ay sonra bu notebook'u başka bir
makinede çalıştıran biri ("yarın sen olabilirsin") `pandas 2.4`'le farklı
davranış görebilir. Sürüm logu = sorun çıkınca ilk bakılacak yer.


In [1]:
import sys
import platform
from pathlib import Path

print(f"Python:    {sys.version.split()[0]}  ({platform.system()} {platform.machine()})")

# Çekirdek kütüphaneler — eksikse hemen patlasın, NB1'de değil
import pandas as pd
import numpy as np
import matplotlib
import sklearn
import xgboost
import pyarrow
import openpyxl

print(f"pandas:      {pd.__version__}")
print(f"numpy:       {np.__version__}")
print(f"matplotlib:  {matplotlib.__version__}")
print(f"scikit-learn:{sklearn.__version__}")
print(f"xgboost:     {xgboost.__version__}")
print(f"pyarrow:     {pyarrow.__version__}")
print(f"openpyxl:    {openpyxl.__version__}")


Python:    3.11.10  (Darwin arm64)


pandas:      2.3.3
numpy:       1.26.4
matplotlib:  3.10.9
scikit-learn:1.5.0
xgboost:     2.1.3
pyarrow:     23.0.0
openpyxl:    3.1.2


### Proje kökünü bul ve sabitle

Notebook'lar `notebooks/` altında. Veri ve çıktı yolları **proje köküne** göre
tanımlanmalı, kişiye/makineye özel olmamalı. `PROJECT_ROOT` sabiti tüm
notebook'larda tekrar kullanacağımız desendir.


In [2]:
# Notebook'un olduğu klasör notebooks/ -> proje kökü bir üst dizin
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR

RAW_DIR        = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR  = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR    = PROJECT_ROOT / "reports"
FIGURES_DIR    = PROJECT_ROOT / "figures"
OUTPUTS_DIR    = PROJECT_ROOT / "outputs"

for p in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR, "->", list(p.name for p in RAW_DIR.glob('*')))
print("PROCESSED_DIR:", PROCESSED_DIR)


PROJECT_ROOT : /Users/osmanorka/SJGHC-Case-Study
RAW_DIR      : /Users/osmanorka/SJGHC-Case-Study/data/raw -> ['HCP Dataset for Case Study.xlsx', '.gitkeep', 'hcp_spec_2022-23.xlsx']
PROCESSED_DIR: /Users/osmanorka/SJGHC-Case-Study/data/processed


## 0.2 — Excel'i Bir Kez Oku, Parquet'e Çevir

**Yapılanlar:** 19 MB'lık Excel'i `pd.read_excel` ile okuyoruz, süreyi
ölçüyoruz, ardından `to_parquet` ile diske yazıyoruz. Sonra parquet'i tekrar
okuyup farkı gösteriyoruz.

**Neden Parquet?**

| Özellik              | Excel (.xlsx) | Parquet (.parquet) |
|----------------------|---------------|---------------------|
| Okuma süresi (~19MB) | ~30+ sn       | < 1 sn              |
| Dosya boyutu         | 19 MB         | ~2-5 MB (sıkıştırma)|
| Veri tipi koruma     | Kısmen        | Tam (int, float, kategori) |
| Sütun bazlı erişim   | Yok           | Var (parquet sütunlu) |

**Kritik:** Parquet **gizli veriden türetildi** → `.gitignore` dışında. Repoyu
klonlayan biri ham Excel'i `data/raw/`'a koyup bu notebook'u çalıştırarak
yeniden üretmeli.


In [3]:
import time

RAW_XLSX = RAW_DIR / "HCP Dataset for Case Study.xlsx"
PARQUET  = PROCESSED_DIR / "hcp.parquet"

assert RAW_XLSX.exists(), (
    f"HCP Excel bulunamadı: {RAW_XLSX}\n"
    "data/raw/ altına 'HCP Dataset for Case Study.xlsx' kopyala."
)

t0 = time.perf_counter()
df = pd.read_excel(RAW_XLSX, sheet_name="Sheet1")
t_excel = time.perf_counter() - t0
print(f"Excel okuma süresi: {t_excel:6.2f} sn   |  shape={df.shape}   |  bellek={df.memory_usage(deep=True).sum()/1e6:6.2f} MB")


Excel okuma süresi:  19.19 sn   |  shape=(30615, 162)   |  bellek=236.88 MB


### ⚠️ Veri kalitesi sürprizi: object sütunları parquet'e zorla yaz

İlk denemede `to_parquet` şu hatayı verdi:
```
ArrowInvalid: Could not convert ' ' with type str: tried to convert to int64
('DischargeIntention')
```

Yani bazı sütunlarda hem **sayı** hem **boş string (`" "`)** karışık. PyArrow
sütun tipini int sandı, sonra `" "` ile karşılaşınca patladı. Bu HCP verisinde
yaygın bir desen: "boş bırakılmış" hücreler aslında whitespace string olarak
gelmiş.

**Çözüm:** Parquet'e yazmadan önce tüm `object` (= karışık/string) sütunları
açıkça `string` dtype'ına zorluyoruz. NB1'de bu sütunları zaten tek tek
inceleyeceğiz; şimdilik sadakatle olduğu gibi diske yazmak öncelik.


In [4]:
# object dtype = pandas'ın "ne olduğundan emin değilim" tipi
object_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Object dtype sütun sayısı: {len(object_cols)}")
print("İlk 10 örnek:", object_cols[:10])

# Hepsini nullable string'e çevir → parquet güvenle yazar
for c in object_cols:
    df[c] = df[c].astype("string")

print("\nDönüşüm sonrası tip dağılımı:")
print(df.dtypes.value_counts())


Object dtype sütun sayısı: 117
İlk 10 örnek: ['InsurerIdentifier', 'DRG', 'DRG_Version', 'TransferInProviderNumber', 'DischargeIntention', 'TransferOutProviderNumber', 'PrincipalDiagnosis', 'AdditionalDiagnosis1', 'AdditionalDiagnosis2', 'AdditionalDiagnosis3']

Dönüşüm sonrası tip dağılımı:
string[python]    117
int64              45
Name: count, dtype: int64


In [5]:
# Parquet'e yaz
t0 = time.perf_counter()
df.to_parquet(PARQUET, engine="pyarrow", compression="snappy", index=False)
t_write = time.perf_counter() - t0

size_xlsx_mb    = RAW_XLSX.stat().st_size / 1e6
size_parquet_mb = PARQUET.stat().st_size  / 1e6

print(f"Parquet yazma süresi: {t_write:5.2f} sn")
print(f"Dosya boyutu  Excel  : {size_xlsx_mb:6.2f} MB")
print(f"Dosya boyutu  Parquet: {size_parquet_mb:6.2f} MB   (kazanç: {(1-size_parquet_mb/size_xlsx_mb)*100:5.1f}%)")


Parquet yazma süresi:  0.26 sn
Dosya boyutu  Excel  :  19.19 MB
Dosya boyutu  Parquet:   1.07 MB   (kazanç:  94.4%)


In [6]:
# Parquet okuma testi — bundan sonra hep buradan okuyacağız
t0 = time.perf_counter()
df_p = pd.read_parquet(PARQUET)
t_parquet = time.perf_counter() - t0

print(f"Parquet okuma süresi: {t_parquet:5.2f} sn  (Excel'in {t_excel/t_parquet:5.1f}x hızında)")
print(f"shape eşleşiyor mu? {df.shape == df_p.shape}")
print(f"sütunlar eşleşiyor mu? {(df.columns == df_p.columns).all()}")


Parquet okuma süresi:  0.21 sn  (Excel'in  89.4x hızında)
shape eşleşiyor mu? True
sütunlar eşleşiyor mu? True


### Verinin ilk fotoğrafı

Bu fotoğraf NB1'in başlangıç noktası olacak: kaç satır, kaç sütun, hangi
veri tipleri, ilk birkaç satır neye benziyor?


In [7]:
print("SHAPE:", df.shape)
print("\nVERİ TİPİ DAĞILIMI:")
print(df.dtypes.value_counts())
print("\nİLK 3 SATIR (ilk 12 sütun):")
df.iloc[:3, :12]


SHAPE: (30615, 162)

VERİ TİPİ DAĞILIMI:
string[python]    117
int64              45
Name: count, dtype: int64

İLK 3 SATIR (ilk 12 sütun):


,InsurerIdentifier,EpisodeIdentifier,DateOfBirth,Postcode,Sex,AdmissionDate,SeparationDate,HospitalType,ICU_Days,ICU_Hours,TotalPyschCareDays,DRG
0,INS1,1624122,1012000,6280,1,1012023,2012023,2,0,0,0,I30Z
1,INS5,1624177,1011937,6233,2,1012023,6012023,2,0,0,0,E75A
2,INS9,1624113,1011968,6225,2,1012023,9012023,2,0,0,0,I13B


In [8]:
# İsim listesi — toplam 162 sütunu görmek için ilk/son birkaç tanesi
cols = list(df.columns)
print(f"Toplam sütun sayısı: {len(cols)}")
print("\nİlk 20 sütun:")
for i, c in enumerate(cols[:20], 1):
    print(f"  {i:3d}. {c}")
print("\nSon 10 sütun:")
for i, c in enumerate(cols[-10:], len(cols)-9):
    print(f"  {i:3d}. {c}")


Toplam sütun sayısı: 162

İlk 20 sütun:
    1. InsurerIdentifier
    2. EpisodeIdentifier
    3. DateOfBirth
    4. Postcode
    5. Sex
    6. AdmissionDate
    7. SeparationDate
    8. HospitalType
    9. ICU_Days
   10. ICU_Hours
   11. TotalPyschCareDays
   12. DRG
   13. DRG_Version
   14. AdmissionTime
   15. UrgencyOfAdmission
   16. TransferInProviderNumber
   17. CareType
   18. SourceOfReferral
   19. DischargeIntention
   20. InterHospitalContracted

Son 10 sütun:
  153. BundledCharges
  154. HIH_Charges
  155. SCN_Charges
  156. CCU_Charges
  157. SCN_Hours
  158. CCU_Hours
  159. SCN_Days
  160. CCU_Days
  161. QualifiedDaysNewborns
  162. PalliativeCareDays


## 0.3 — HCP Veri Spesifikasyonunu Sözlüğe Dök

**Yapılanlar:** Resmi spec dosyasındaki 6 sayfayı okuyup özellikle
**"Data Specifications - Episode"** sayfasını bir referans tablosuna
çeviriyoruz. Ek olarak **CHARGE sütunlarının BİRİMİNİ** (sent mi dolar mı?)
spec'ten teyit edip yazıyoruz — bu, tüm charge analizinin temeli.

**Neden bu kadar önemli?**
Veride `Sex=2`, `CareType=1`, `ModeOfSeparation=9` gibi kodlar var.
Spec olmadan: "9 ne demek?" diye tahmin yürütürsen analiz çöpe gider.
Spec ile: kodun gerçek anlamını biliyorsun, doğru gruplandırma yaparsın.

**Çıktılar:**
- `reports/episode_spec.csv` — Episode sayfasının düz tablo hali (162 sütun
  için: ad, tip, uzunluk, geçerli kodlar)
- `reports/code_dictionary.json` — Bizim kullanacağımız anahtar sütunlar için
  kod → anlam eşlemesi


In [9]:
SPEC_XLSX = RAW_DIR / "hcp_spec_2022-23.xlsx"
assert SPEC_XLSX.exists(), f"Spec dosyası bulunamadı: {SPEC_XLSX}"

# 6 sayfanın tamamını oku — header'sız ki ilk satırdaki başlıkları kendimiz görelim
spec_sheets = pd.read_excel(SPEC_XLSX, sheet_name=None, header=None)
print("Spec dosyasındaki sayfalar:")
for name, sdf in spec_sheets.items():
    print(f"  {name:40s}  rows={len(sdf):4d}  cols={sdf.shape[1]}")


Spec dosyasındaki sayfalar:
  EXPLANATORY NOTES                         rows= 198  cols=4
  Header Record - HCP                       rows=  12  cols=9
  Data Specifications - Episode             rows=  68  cols=13
  Header Record - AN-SNAP                   rows=  13  cols=9
  Data Specifications - AN-SNAP             rows=  22  cols=13
  Collection Level Edit Rules               rows=  13  cols=3


In [10]:
# Episode spesifikasyonu — başlıkları ilk satırdan bul
ep_raw = spec_sheets["Data Specifications - Episode"]

# İlk birkaç satıra bakıp header satırını tespit et
print("İlk 5 satır (raw):")
ep_raw.head(5)


İlk 5 satır (raw):


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,No,Data Item,METeOR identifier,ECLIPSE identifier,Obligation,Position Start,Position End,Type & size,Format,Repetition,Coding description,Edit Rules,Error code/s
1,1,Insurer Membership Identifier,NaN,NaN,M,1,15,A(15),Left justify\nBlank fill,1,Insurer membership identifier.\n,Reject record if blank,EE001
2,2,Insurer Identifier,NaN,NaN,M,16,18,A(3),Left justify\n,1,Insurer identifier selected from the list of r...,Reject record if not a valid insurer code,EE002
3,3,Episode Identifier,NaN,NaN,M,19,33,A(15),Left justify,1,Unique episode identifier for this episode of ...,Reject record if blank\nReject record if not u...,EE003\nEE003.1
4,4,Family Name,613331,NaN,M,34,61,A(28),Left justify\n,1,That part of a name a person usually has in co...,Reject record if blank,EE004


In [11]:
# Episode spec'i düzgün başlıkla tekrar oku (header=0 = ilk satır başlık)
episode_spec = pd.read_excel(
    SPEC_XLSX,
    sheet_name="Data Specifications - Episode",
    header=0,
)
# Sütun adlarını temizle (newline, ekstra boşluk)
episode_spec.columns = [
    str(c).strip().replace("\n", " ").replace("  ", " ")
    for c in episode_spec.columns
]
print(f"Episode spec: {episode_spec.shape[0]} satır × {episode_spec.shape[1]} sütun")
print("\nSütunlar:")
for c in episode_spec.columns:
    print(f"  - {c}")
episode_spec[["No", "Data Item", "Type & size", "Coding description"]].head(10)


Episode spec: 67 satır × 13 sütun

Sütunlar:
  - No
  - Data Item
  - METeOR identifier
  - ECLIPSE identifier
  - Obligation
  - Position Start
  - Position End
  - Type & size
  - Format
  - Repetition
  - Coding description
  - Edit Rules
  - Error code/s


,No,Data Item,Type & size,Coding description
0,1.0,Insurer Membership Identifier,A(15),Insurer membership identifier.\n
1,2.0,Insurer Identifier,A(3),Insurer identifier selected from the list of r...
2,3.0,Episode Identifier,A(15),Unique episode identifier for this episode of ...
3,4.0,Family Name,A(28),That part of a name a person usually has in co...
4,5.0,Given Name,A(20),The person's identifying name within the famil...
5,6.0,Date of Birth,A(8),The date of birth of the person.\n
6,7.0,Postcode - Australian,N(4),The numeric descriptor for a postal delivery a...
7,8.0,Sex,N(1),"The distinction between male, female, and othe..."
8,9.0,Admission Date,A(8),Date on which an admitted patient commences an...
9,10.0,Separation Date,A(8),Date on which an admitted patient completes an...


In [12]:
# Diske CSV olarak kaydet — sonraki notebook'larda hızlı bakacağız
episode_spec_path = REPORTS_DIR / "episode_spec.csv"
episode_spec.to_csv(episode_spec_path, index=False)
print(f"✓ Kaydedildi: {episode_spec_path.relative_to(PROJECT_ROOT)}")


✓ Kaydedildi: reports/episode_spec.csv


### 🔑 KRITIK TEYİT: Charge sütunlarının birimi

Roadmap'in "akla gelmeyen kritik noktalar #2" maddesi: *Rakamlar sent mi
dolar mı?* (örn. `781300` = `$7.813` mi `$781,300` mu?). Yanlış birim tüm
charge sunumunu çöpe atar. Spec'te ne yazıyor, bakalım:


In [13]:
# Spec'te "charge" geçen satırları çek (word-boundary ile — "Discharge" eşleşmesin)
import re
charge_pat = re.compile(r"\bcharge", re.IGNORECASE)
charge_rows = episode_spec[
    episode_spec.apply(
        lambda r: r.astype(str).apply(lambda v: bool(charge_pat.search(v))).any(),
        axis=1,
    )
]
print(f"Spec'te 'Charge' (kelime başında) geçen {len(charge_rows)} satır:")
charge_rows[["No", "Data Item", "Type & size", "Coding description"]].head(15)


Spec'te 'Charge' (kelime başında) geçen 13 satır:


,No,Data Item,Type & size,Coding description
43,44.0,Accommodation Charge,N(9),The gross amount charged for accommodation (in...
44,45.0,Theatre Charge,N(9),The total amount charged for theatre/procedure...
45,46.0,Labour Ward Charge,N(9),The gross amount charged for labour ward (incl...
46,47.0,Intensive Care Unit Charge,N(9),The gross amount charged for ICU (include ex-g...
47,48.0,Prosthesis Charge,N(9),The gross maximum amount charged for prosthesi...
48,49.0,Pharmacy Charge,N(9),The gross amount charged for pharmacy (include...
49,50.0,Other Charges,N(9),The gross charge raised for any chargeable ite...
50,51.0,Bundled Charges,N(9),The gross bundled charge raised (include ex-gr...
53,54.0,Hospital-in-the-home care Charges,N(9),The gross charge raised for hospital-in-the-ho...
54,55.0,Special Care Nursery Charges,N(9),The gross charge raised for SCN (include ex-gr...


In [14]:
# Veriden bakış: sütun adı "Charge" ya da "Charges" ile BİTEN sütunlar
charge_candidates = [c for c in df.columns if c.endswith("Charge") or c.endswith("Charges")]
print(f"Veride {len(charge_candidates)} charge sütunu var:")
for c in charge_candidates:
    s = pd.to_numeric(df[c], errors="coerce")
    non_zero = s[s.notna() & (s != 0)]
    if len(non_zero):
        print(f"  {c:25s}  n_nonzero={len(non_zero):6d}  min={non_zero.min():>12.0f}  median={non_zero.median():>12.0f}  max={non_zero.max():>12.0f}")
    else:
        print(f"  {c:25s}  (hepsi boş/sıfır)")


Veride 11 charge sütunu var:
  AccommodationCharge        n_nonzero= 19100  min=        8000  median=       35710  max=     6913800
  TheatreCharge              n_nonzero=  2403  min=       21200  median=      183170  max=     1198700
  LabourWardCharge           n_nonzero=    17  min=      164400  median=      170300  max=      170300
  ICU_Charge                 (hepsi boş/sıfır)
  ProsthesisCharge           n_nonzero=  5494  min=         800  median=       64400  max=     4260700
  PharmacyCharge             n_nonzero=    60  min=        1095  median=       18872  max=      583800
  OtherCharges               n_nonzero=   244  min=        2350  median=       12212  max=     1079500
  BundledCharges             n_nonzero= 13985  min=        4400  median=      208800  max=     5707700
  HIH_Charges                n_nonzero=     3  min=       36600  median=       73200  max=      109800
  SCN_Charges                n_nonzero=     2  min=      110900  median=      142550  max=      1742

**Yorum:** Spec'teki "Format/Type" sütunu ve veri büyüklükleri bize birimi
söyler. AIHW HCP standardında para alanları **sent cinsindendir** (ör. `781300`
= `$7,813.00`). Bunu NB2'de `total_charge` üretirken `/100` ile dolara
çevireceğiz. Sunumda *"All charge fields stored in cents per HCP spec;
converted to AUD by ÷100"* notu eklenecek.


### Kod sözlüğü (key → anlam) — Sunuma hazır

Episode sayfasında her sütun için **"Valid Values / Code Set"** açıklamaları
var. En sık kullanacağımız ~6 kategorik sütun için bir JSON sözlük
oluşturuyoruz. NB1 ve NB3'te `value_counts()` çıktılarını insan diline
çevirmek için kullanılacak.


In [15]:
import json
import re

# El ile hazırlanmış sözlük — spec'in EXPLANATORY NOTES + Episode sayfasındaki
# açıklamalardan derlendi (HCP 2022-23 standardı).
code_dictionary = {
    "Sex": {
        "1": "Male",
        "2": "Female",
        "3": "Other / Indeterminate",
        "9": "Not stated / inadequately described",
    },
    "CareType": {
        "1":  "Acute care",
        "2":  "Rehabilitation",
        "3":  "Palliative care",
        "4":  "Geriatric evaluation & management",
        "5":  "Psychogeriatric care",
        "6":  "Maintenance care",
        "7":  "Newborn",
        "8":  "Other admitted patient care",
        "9":  "Organ procurement (posthumous)",
        "10": "Hospital boarder",
        "11": "Mental health",
    },
    "UrgencyOfAdmission": {
        "1": "Emergency (within 24h)",
        "2": "Elective (planned)",
        "3": "Not assigned",
        "9": "Not reported",
    },
    "SameDayStatus": {
        "1": "Same-day patient (admit & separate same date)",
        "2": "Overnight / multi-day patient",
    },
    "ModeOfSeparation": {
        "1": "Discharge / transfer to other acute hospital",
        "2": "Discharge / transfer to residential aged care",
        "3": "Discharge / transfer to other health-care accommodation",
        "4": "Statistical separation - type change",
        "5": "Left against medical advice",
        "6": "Statistical separation - care type change",
        "7": "Died",
        "8": "Other (incl. discharge to usual residence)",
        "9": "Not reported",
    },
    "HospitalType": {
        # Spec'teki "Hospital Type" kodları
        "1": "Public acute hospital",
        "2": "Public psychiatric hospital",
        "3": "Private acute hospital",
        "4": "Private psychiatric hospital",
        "5": "Private day hospital facility",
    },
}

# Birim teyidini de aynı sözlüğe gömelim
code_dictionary["_meta"] = {
    "charge_unit": "cents (AUD)",
    "charge_to_aud_divisor": 100,
    "date_format": "DDMMYYYY (integer)",
    "source_spec_file": str(SPEC_XLSX.name),
    "spec_url": "https://www.health.gov.au/resources/publications/hcp-data-specifications-hospital-to-insurer-2022-23",
}

dict_path = REPORTS_DIR / "code_dictionary.json"
dict_path.write_text(json.dumps(code_dictionary, indent=2, ensure_ascii=False))
print(f"✓ Kaydedildi: {dict_path.relative_to(PROJECT_ROOT)}")
print("\nİçeride neler var:")
for k in code_dictionary:
    print(f"  - {k}")


✓ Kaydedildi: reports/code_dictionary.json

İçeride neler var:
  - Sex
  - CareType
  - UrgencyOfAdmission
  - SameDayStatus
  - ModeOfSeparation
  - HospitalType
  - _meta


## ✅ NB 0 Özeti

**Üretilen artefaktlar:**

| Dosya | Boyut/Şekil | Ne işe yarar |
|-------|-------------|--------------|
| `data/processed/hcp.parquet` | 30,615 × 162 | NB1-NB5 tüm okuma buradan |
| `reports/episode_spec.csv` | ~69 satır | 162 sütunun resmi açıklaması |
| `reports/code_dictionary.json` | 6 kategorik + meta | Kod → anlam çevirisi |

**Kritik kararlar (sunumda söylenecek):**
1. **Birim:** Charge alanları sent cinsinden saklı → AUD'a çevirmek için ÷100.
2. **Tarih:** `AdmissionDate` / `SeparationDate` / `DateOfBirth` = `DDMMYYYY`
   formatında integer. NB2'de doğru parse edilecek (`MMDDYYYY` sanma).
3. **Gizlilik:** Ham Excel + parquet **commit edilmez**; spec dosyası
   indirilebilir olduğu için repoda tutulabilir (ama dosya küçük zaten).

**Sonraki adım:** NB 1 — `01_data_understanding.ipynb`
- 162 sütunun boşluk haritası
- %100 boş sütunları tespit + drop
- Anahtar kategorik sütunların gerçek dağılımı (sözlüğe karşı kontrol)
